In [26]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

def add_features(df):
    df = df.copy()
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["IsChild"] = (df["Age"] < 13).astype(int)
    df["FarePerPerson"] = df["Fare"] / df["FamilySize"]
    return df

train = add_features(train)
test = add_features(test)

num = ["Age", "Fare", "SibSp", "Parch", "FamilySize", "IsAlone", "IsChild", "FarePerPerson"]
cat = ["Pclass", "Sex", "Embarked"]

X_train = train[num + cat]
y_train = train["Survived"]
X_test = test[num + cat]

def make_preprocess(scale):
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        num_steps.append(("scaler", StandardScaler()))

    return ColumnTransformer([
        ("num", Pipeline(num_steps), num),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore")),
        ]), cat),
    ])

In [27]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

baseline = Pipeline([
    ("prep", make_preprocess(False)),
    ("model", DummyClassifier(strategy="constant", constant=1))
])

tree = Pipeline([
    ("prep", make_preprocess(False)),
    ("model", DecisionTreeClassifier(
        max_depth=4,
        min_samples_leaf=10,
        random_state=27
    ))
])

forest = Pipeline([
    ("prep", make_preprocess(False)),
    ("model", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        random_state=27,
        n_jobs=-1
    ))
])

boosting = Pipeline([
    ("prep", make_preprocess(False)),
    ("model", GradientBoostingClassifier(random_state=27))
])


svm = Pipeline([
    ("prep", make_preprocess(True)),
    ("model", CalibratedClassifierCV(
        SVC(C=1.0, kernel="rbf", gamma="scale"),
        ensemble=False
    ))
])

knn = Pipeline([
    ("prep", make_preprocess(True)),
    ("model", KNeighborsClassifier(
        n_neighbors=7,
        weights="distance"
    ))
])

In [28]:
from sklearn.metrics import f1_score
from sklearn.base import clone
import numpy as np

models = {
    "Always 1": baseline,
    "Decision Tree": tree,
    "Random Forest": forest,
    "Gradient Boosting": boosting,
    "SVC": svm,
    "KNN": knn,
}

def predict_with_threshold(pipeline, X, threshold=0.5):
    probabilities = pipeline.predict_proba(X)[:, 1]
    predictions = (probabilities >= threshold).astype(int)
    return predictions

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=27)

thresholds = np.arange(0.25, 0.56, 0.05) 
best_thresholds = {}
best_cv_scores = {}
print("Поиск оптимального порога на кросс-валидации (F1-score):")
print("-" * 75)

for name, model in models.items():
    all_fold_probs = []
    all_fold_y_true = []
    
    
    for train_idx, val_idx in cv.split(X_train, y_train):
        fitted_model = clone(model)
        
        X_tr = X_train.iloc[train_idx]
        y_tr = y_train.iloc[train_idx]
        X_va = X_train.iloc[val_idx]
        y_va = y_train.iloc[val_idx] 
        
        fitted_model.fit(X_tr, y_tr)
        
        probs = fitted_model.predict_proba(X_va)[:, 1]
        all_fold_probs.append(probs)
        all_fold_y_true.append(y_va)
        
    best_score = -1
    best_thresh = 0.5
    best_scores_std = 0.0
    
    for t in thresholds:
        fold_scores = []
        # Считаем f1 для каждого фолда при текущем пороге t
        for probs, y_true in zip(all_fold_probs, all_fold_y_true):
            y_pred = (probs >= t).astype(int)
            score = f1_score(y_true, y_pred)
            fold_scores.append(score)
            
        mean_score = np.mean(fold_scores)
        std_score = np.std(fold_scores)
        
        # Ищем максимум среднего F1
        if mean_score > best_score:
            best_score = mean_score
            best_thresh = t
            best_scores_std = std_score
            
    best_thresholds[name] = best_thresh
    best_cv_scores[name] = best_score
    
    print(f"{name:<25} Лучший порог: {best_thresh:.2f}  |  F1: {best_score:.3f} ± {best_scores_std:.3f}")



    

Поиск оптимального порога на кросс-валидации (F1-score):
---------------------------------------------------------------------------
Always 1                  Лучший порог: 0.25  |  F1: 0.555 ± 0.002
Decision Tree             Лучший порог: 0.30  |  F1: 0.741 ± 0.026
Random Forest             Лучший порог: 0.35  |  F1: 0.758 ± 0.030
Gradient Boosting         Лучший порог: 0.35  |  F1: 0.769 ± 0.025
SVC                       Лучший порог: 0.45  |  F1: 0.755 ± 0.033
KNN                       Лучший порог: 0.45  |  F1: 0.724 ± 0.039


Финальное обучение на всём X_train и предсказание на X_test:


In [ ]:
best_name = max(best_cv_scores, key=best_cv_scores.get)
best = clone(models[best_name]).fit(X_train, y_train)

submission = pd.DataFrame({
    "Id": test["Id"],
    "Survived": predict_with_threshold(best, X_test, best_thresholds[best_name]),
})
submission.to_csv("submission.csv", index=False)
print(best_name, best_thresholds[best_name])


## Качество моделей

Тестовых ответов нет, поэтому метрики считаются на `train` по out-of-fold предсказаниям: каждого пассажира оценивает модель, обученная без него. Порог взят из подбора выше.

In [ ]:
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
import matplotlib.pyplot as plt

# Каждый пассажир получает предсказание от модели, которая его не видела при обучении
oof_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=27)

oof_probs = {}
oof_preds = {}
rows = []

for name, model in models.items():
    probs = cross_val_predict(model, X_train, y_train, cv=oof_cv, method="predict_proba")[:, 1]
    preds = (probs >= best_thresholds[name]).astype(int)
    oof_probs[name] = probs
    oof_preds[name] = preds

    tn, fp, fn, tp = confusion_matrix(y_train, preds).ravel()
    rows.append({
        "Model": name,
        "Threshold": round(float(best_thresholds[name]), 2),
        "Accuracy": accuracy_score(y_train, preds),
        "Precision": precision_score(y_train, preds),
        "Recall": recall_score(y_train, preds),
        "F1": f1_score(y_train, preds),
        "FP": fp,
        "FN": fn,
    })

metrics_table = pd.DataFrame(rows).set_index("Model").sort_values("F1", ascending=False)
metrics_table.round(3)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7.5))

for ax, name in zip(axes.ravel(), metrics_table.index):
    ConfusionMatrixDisplay.from_predictions(
        y_train,
        oof_preds[name],
        display_labels=["Погиб", "Выжил"],
        cmap="Blues",
        colorbar=False,
        ax=ax,
    )
    ax.set_title(f"{name} (порог {best_thresholds[name]:.2f})")
    ax.set_xlabel("Предсказание")
    ax.set_ylabel("Истина")

plt.tight_layout()
plt.show()

## Важность признаков

In [ ]:
from sklearn.inspection import permutation_importance

# Перемешиваем один столбец и смотрим, насколько упал F1 на данных вне обучения.
# Способ работает для всех моделей, включая SVC и KNN.
def fold_permutation_importance(model, threshold, n_repeats=5):
    def scorer(estimator, X, y):
        return f1_score(y, predict_with_threshold(estimator, X, threshold))

    fold_results = []
    for train_idx, val_idx in oof_cv.split(X_train, y_train):
        fitted = clone(model).fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
        result = permutation_importance(
            fitted,
            X_train.iloc[val_idx],
            y_train.iloc[val_idx],
            scoring=scorer,
            n_repeats=n_repeats,
            random_state=27,
        )
        fold_results.append(result.importances_mean)

    return pd.Series(np.mean(fold_results, axis=0), index=X_train.columns)

importances = pd.DataFrame({
    name: fold_permutation_importance(model, best_thresholds[name])
    for name, model in models.items()
    if name != "Always 1"
})

model_names = [name for name in metrics_table.index if name != "Always 1"]
fig, axes = plt.subplots(2, 3, figsize=(13, 8), sharex=True)

for ax, name in zip(axes.ravel(), model_names):
    values = importances[name].sort_values()
    ax.barh(values.index, values.values, color="#2a6fb0", height=0.6)
    ax.axvline(0, color="#777777", linewidth=1)
    ax.set_title(name)
    ax.set_xlabel("Падение F1 после перемешивания")
    ax.spines[["top", "right"]].set_visible(False)

axes.ravel()[-1].set_visible(False)
plt.tight_layout()
plt.show()

importances.loc[importances.mean(axis=1).sort_values(ascending=False).index].round(3)

## Анализ ошибок лучшей модели

FP: модель сказала «выжил», а пассажир погиб. FN: модель сказала «погиб», а пассажир выжил.

In [ ]:
analysis = train.copy()
analysis["prob"] = oof_probs[best_name]
analysis["pred"] = oof_preds[best_name]

analysis["error"] = "верно"
analysis.loc[(analysis["pred"] == 1) & (analysis["Survived"] == 0), "error"] = "FP"
analysis.loc[(analysis["pred"] == 0) & (analysis["Survived"] == 1), "error"] = "FN"

analysis["Class"] = analysis["Pclass"].astype(str) + " класс"
analysis["SexClass"] = analysis["Sex"].astype(str) + ", " + analysis["Class"]
analysis["AgeGroup"] = (
    pd.cut(
        analysis["Age"],
        bins=[0, 12, 17, 35, 60, 100],
        labels=["0-12", "13-17", "18-35", "36-60", "61+"],
    )
    .astype(object)
    .fillna("неизвестен")
)
analysis["FareGroup"] = pd.qcut(
    analysis["Fare"], 4, labels=["Q1 дешёвые", "Q2", "Q3", "Q4 дорогие"]
).astype(object)
analysis["Family"] = analysis["IsAlone"].map({1: "один", 0: "с семьёй"})
analysis["Port"] = analysis["Embarked"].fillna("неизвестен")

print(analysis["error"].value_counts().to_string())

def group_report(column):
    report = analysis.groupby(column).agg(
        passengers=("error", "size"),
        survived_share=("Survived", "mean"),
        FP=("error", lambda s: (s == "FP").sum()),
        FN=("error", lambda s: (s == "FN").sum()),
    )
    report["error_rate"] = (report["FP"] + report["FN"]) / report["passengers"]
    report.insert(0, "feature", column)
    return report

group_table = pd.concat([
    group_report(column)
    for column in ["Sex", "Class", "SexClass", "AgeGroup", "FareGroup", "Family", "Port"]
])

group_table.sort_values("error_rate", ascending=False).round(3)

### Примеры уверенных ошибок

In [ ]:
# Похожие пассажиры: тот же пол, класс и возрастная группа, без самого объекта
similar = analysis.groupby(["SexClass", "AgeGroup"])["Survived"]
analysis["similar_count"] = similar.transform("size") - 1
analysis["similar_survived"] = (
    (similar.transform("sum") - analysis["Survived"]) / analysis["similar_count"]
)

columns = [
    "Id", "Sex", "Pclass", "Age", "SibSp", "Parch", "Fare", "Embarked",
    "Survived", "prob", "error", "similar_count", "similar_survived",
]

confident_fp = analysis[analysis["error"] == "FP"].nlargest(4, "prob")
confident_fn = analysis[analysis["error"] == "FN"].nsmallest(4, "prob")

examples = pd.concat([confident_fp, confident_fn])[columns]
examples.round(3)

### Возможные причины ошибок

`similar_survived` показывает долю выживших среди пассажиров того же пола, класса и возрастной группы, `similar_count` показывает их число. Если эта доля близка к прогнозу модели, значит, модель верно усвоила общее правило, а пассажир оказался исключением.

| Id | Ошибка | Кто это | Вероятная причина |
|---|---|---|---|
| 14438670 | FP | Женщина, 1 класс, 50 лет, одна | Все 35 похожих пассажирок выжили. Признаки не отличают её от них |
| 33509580 | FP | Девочка, 1 класс, 2 года | Похожих в данных нет. Такой же тариф 151.55 у ещё трёх пассажиров, возможно, это одна семья или один билет. Модель смотрит на каждого отдельно и не знает судьбу группы |
| 89000739 | FP | Женщина, 1 класс, 25 лет | Та же ситуация: тариф 151.55, те же SibSp и Parch. Все 38 похожих пассажирок выжили |
| 10000257 | FP | Женщина, 2 класс, 57 лет, одна | Около 90% похожих выжили. Это редкое исключение |
| 92582288 | FN | Мужчина, 3 класс, 25 лет, тариф 0 | Выжили только 15% похожих, а среди 15 пассажиров с нулевым тарифом выжил один |
| 10000123 | FN | Мужчина, 3 класс, 45 лет, один | Выжили менее 5% похожих. Причин спасения нет в признаках: неизвестны каюта, палуба и место на корабле в момент эвакуации |
| 10000220 | FN | Мужчина, 3 класс, 44 года, один | То же самое |
| 56211872 | FN | Женщина, 3 класс, 38 лет, с шестью родственниками | Модель усвоила, что большие семьи 3 класса почти не выживали: 11%. Такой же тариф у детей с похожим составом семьи, поэтому это, вероятно, одна семья с разной судьбой |

Общий вывод: уверенные ошибки вызваны не поломкой модели, а тем, что эти пассажиры нарушают правило своей группы. По имеющимся столбцам отличить их от похожих пассажиров нельзя.